In [20]:
import os
import dotenv
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate

dotenv.load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY1")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL")

llm=ChatOpenAI(
    # base_url=os.getenv('OPENAI_BASE_URL'),
    # api_key=os.getenv('OPENAI_API_KEY1'),
    model='gpt-4o-mini',
    temperature=0.8,
    max_tokens=40
)

In [2]:
res=llm.invoke('什么是大语言模型')
print(type(res)) #AIMessage

<class 'langchain_core.messages.ai.AIMessage'>


一、字符串

In [7]:
#1.content
print(res.content)

大语言模型（Large Language Model，LLM）是一种基于深度学习技术的


In [8]:
#2.StrOutputParser
parser=StrOutputParser()
str_res=parser.invoke(res)
print(type(str_res))
print(str_res)

<class 'str'>
大语言模型（Large Language Model，LLM）是一种基于深度学习技术的


二、json

In [19]:
#1.JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate

model = ChatOpenAI(
    model='gpt-4o-mini',
)

chat_prompt_template = ChatPromptTemplate.from_messages([('system', '你是一个{role}'),
                                                         ('human', '{question}'), ])
#方式一、需要自己在提示词中告诉大模型返回json
prompt_res = chat_prompt_template.invoke(
    {'role': 'AI专家', 'question': '人工智能的英文怎么说?问题用q表示,答案用a表示 返回一个json格式数据'})
res = model.invoke(prompt_res)
print(res.content)

parser = JsonOutputParser()
json_res = parser.invoke(res)
print(json_res)

```json
{
  "q": "人工智能的英文怎么说?",
  "a": "Artificial Intelligence"
}
```
{'q': '人工智能的英文怎么说?', 'a': 'Artificial Intelligence'}


In [23]:
from langchain_core.prompts import PromptTemplate

#2.不需要手动告诉返回json
model = ChatOpenAI(model='gpt-4o-mini')

joke_query='告诉我一个笑话'

parser = JsonOutputParser()
#利用partial 告诉模型返回json
prompt_template = PromptTemplate.from_template(template='回答用户的查询\n，满足的格式为{format}\n问题为{ques}\n',
                                        partial_variables={'format': parser.get_format_instructions()})

prompt_res=prompt_template.invoke({'ques': joke_query})
res=model.invoke(prompt_res)
print(res.content)

json_res=parser.invoke(res)
print(json_res)

```json
{
  "joke": "为什么数学书总是很坏？因为它有太多的问题！"
}
```
{'joke': '为什么数学书总是很坏？因为它有太多的问题！'}


拓展 管道执行

In [24]:
chain = prompt_template | model | parser
print(chain.invoke(
    {'ques': joke_query}
))

{'joke': '为什么计算机很冷？因为它们总是开着窗口！'}


三、xml 应该用得不多